aa

In [39]:
import dataset.load_dataset as load_dataset
import json
import os
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from collections import defaultdict
import importlib
importlib.reload(load_dataset)

<module 'dataset.load_dataset' from 'c:\\Users\\Balthazar\\Shared\\waste-classification-system-II\\dataset\\load_dataset.py'>

In [40]:
CONFIG_PATH = "dataset/dataset.json"
DATASETS_ROOT = "dataset/"

In [ ]:
# Load the dataset configuration json file
config = load_dataset.load_config(CONFIG_PATH)

# Example usage of the dataset configuration enable/disable datasets, subdatasets, classes, and labels.

# 1. Whole dataset
# load_dataset.enable_dataset(config, "Garbage Dataset")

# 2. One resolution variant / subdataset
# load_dataset.disable_subdataset(config, "Garbage Dataset", "standardized_256")
# load_dataset.disable_subdataset(config, "Garbage Dataset", "standardized_384")

# 3. One class inside a subdataset
# load_dataset.disable_class(config, "Garbage Dataset", "original", "shoes")

# load_dataset.disable_label(config, "Metal")   # turns off metal in every source


# Turn ALL sources of a unified label on/off (across every dataset)
# disable_label(config, "Metal")

# Turn one class inside a subdataset on/off
# disable_class(config, "Garbage Dataset", "original", "shoes")
# enable_class(config,  "Garbage Dataset", "original", "shoes")

# Save changes back to disk (optional — skipping this means changes only last the session)
load_dataset.save_config(config, "dataset/dataset.json")

# print_summary
load_dataset.status(config)


[SAVED] Config written to dataset/dataset.json
[ON ] Garbage Classification Dataset
       [ON ] main
              [ON ] cardboard                 → Cardboard
              [ON ] glass                     → Glass
              [ON ] metal                     → Metal
              [ON ] paper                     → Paper
              [ON ] plastic                   → Plastic
              [ON ] trash                     → Trash
[ON ] Garbage Dataset
       [ON ] original
              [ON ] battery                   → Batteries
              [ON ] biological                → Organic
              [ON ] cardboard                 → Cardboard
              [ON ] clothes                   → Textile
              [ON ] glass                     → Glass
              [ON ] metal                     → Metal
              [ON ] paper                     → Paper
              [ON ] plastic                   → Plastic
              [ON ] shoes                     → Textile
              [ON ] tr

In [ ]:
default_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

config       = load_dataset.load_config(CONFIG_PATH)
samples      = load_dataset.collect_samples(config, datasets_root=DATASETS_ROOT)
label_to_idx, idx_to_label = load_dataset.build_label_maps(samples)

load_dataset.print_summary(samples, label_to_idx)

dataset = load_dataset.WasteDataset(samples, label_to_idx, transform=default_transform)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

images, labels = next(iter(loader))
print(f"Batch shape : {images.shape}") # ex torch.Size([32, 3, 224, 224])
print(f"Label indices: {labels.tolist()[:8]}")
print(f"Label names  : {[idx_to_label[i.item()] for i in labels[:8]]}")

[OK]  Garbage Classification Dataset           / main                      / cardboard             →  Cardboard     (403 images)
[OK]  Garbage Classification Dataset           / main                      / glass                 →  Glass         (501 images)
[OK]  Garbage Classification Dataset           / main                      / metal                 →  Metal         (410 images)
[OK]  Garbage Classification Dataset           / main                      / paper                 →  Paper         (594 images)
[OK]  Garbage Classification Dataset           / main                      / plastic               →  Plastic       (482 images)
[OK]  Garbage Classification Dataset           / main                      / trash                 →  Trash         (137 images)
[OK]  Garbage Dataset                          / original                  / battery               →  Batteries     (756 images)
[OK]  Garbage Dataset                          / original                  / biological          

c:\Users\Balthazar\miniconda3\envs\cpu_env\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Batch shape : torch.Size([32, 3, 224, 224])
Label indices: [4, 8, 2, 6, 1, 3, 2, 6]
Label names  : ['Metal', 'Textile', 'E-waste', 'Paper', 'Cardboard', 'Glass', 'E-waste', 'Paper']
